In [ ]:
from utils import *

import torch as th
import torch.nn as nn
import torch.nn.utils.prune as prune
import torch.nn.functional as F

import importlib
import data_handler

importlib.reload(data_handler)

import tqdm


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class RNet(nn.Module):
    def __init__(self, num_classes=2):
        super(RNet, self).__init__()
        self.resnet = models.resnet18(pretrained=False)
        
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.resnet(x)

# Example usage:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RNet(num_classes=2).to(device)


In [ ]:
import torch
import torch.nn.utils.prune as prune

def apply_pruning(model, sparsity=0.5):
    """
    Applies unstructured pruning to each layer in the model progressively.
    """
    for name, module in model.named_modules():
        if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)):
            prune.l1_unstructured(module, name='weight', amount=sparsity)

def train_and_prune(model, epochs, initial_sparsity=0.0, final_sparsity=0.9):
    """
    Gradually prunes the model over several epochs.
    """
    sparsity_step = (final_sparsity - initial_sparsity) / epochs
    loss_func = nn.CrossEntropyLoss()
    optimizer = th.optim.Adam(model.parameters(), lr=0.0001)

    for epoch in range(epochs):
        current_sparsity = initial_sparsity + epoch * sparsity_step

        # Training code 
        model.train()
        for images, labels in data_handler.tr_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            apply_pruning(model, current_sparsity)
            outputs = model(images)
            loss = loss_func(outputs, labels)
            apply_pruning(model,current_sparsity)
            loss.backward()
            optimizer.step()
        

        # Print progress and validate
        print(f"Epoch {epoch+1}/{epochs}, Sparsity: {current_sparsity:.2f}, Loss: {loss.item()}")

    # Remove pruning re-parametrization to finalize the model's sparsity
    for module in model.modules():
        if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)):
            prune.remove(module, 'weight')

    # Fine-tune the pruned model
    fine_tune_epochs = 5  
    for epoch in range(fine_tune_epochs):
        model.train()
        for images, labels in data_handler.tr_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_func(outputs, labels)
            loss.backward()
            optimizer.step()
        print(f"Fine-tuning Epoch {epoch+1}/{fine_tune_epochs}, Loss: {loss.item()}")
    
    torch.save(model.state_dict(), "resnet18_00_90.pt")


In [ ]:
train_and_prune(model, 10)

In [ ]:
# Load the model
model = RNet().to(device)
#model.load_state_dict(th.load("resnet18_not_pretrained.pt"))
model.load_state_dict(th.load("resnet18_00_90.pt"))

# Test the model
model.eval()

# Test loop
total_images = 0
nr_acc = 0
for images, label in data_handler.te_loader:
    images = images.to(device)
    labels = label.to(device)
    
    # Forward pass
    outputs = model(images)
    
    # Predicted classes
    predicted = th.argmax(outputs, dim=1)
    
    # Accuracy calculation (vectorized)
    nr_acc += (predicted == labels).sum().item()  # Count correct predictions
    total_images += labels.size(0)  # Keep track of the total number of images
    
    
    # Original image
    # plt.imshow(images[0].cpu().permute(1, 2, 0).numpy())
    # plt.title(f"Pred: {"1" if predicted[0] else "0"} | Label: {labels[0]}")
    # plt.axis('off')
    
    # plt.show()

print(nr_acc/total_images)